# 06 — Figures

This notebook draws the paper figures from the atlas, circulating-cell layout, cluster profiles and leukemia
contrasts. It saves the values behind the plots so readers can inspect the numbers as well as the pictures.

Steps:
1. Draw the transferred AML labels and their sources, profile correlations and shared leukemia genes
(F0–F3).
2. Show programme genes, the PM4 signature, actin and pyr expression (F4–F7).
3. Compare actin between stages and cell types within the same samples (F8).

Outputs: SVG with editable text, PDF and 300-dpi PNG in `figures/`, and plotted values in `figures/tables/`.

| figure | what it shows | inputs |
|---|---|---|
| F0 | AML labels on the circulating-cell UMAP, turned 90° clockwise | integration, labels |
| F1 | Non-AML cells of each AML label, by dataset and published label | atlas, labels |
| F2 | Relative-profile correlations, for Yoon and published labels | cluster profiles |
| F3 | Top 30 shared leukemia genes, whole arm and within cluster | leukemia contrasts |
| F4 | Programme marker genes in AML clusters | atlas |
| F5 | PM4 signature: standardized score and score itself | atlas, PM4 signature genes |
| F6 | Act5C and Act42A: standardized expression and mean expression | atlas |
| F7 | pyr: standardized expression and mean expression | atlas |
| F8 | Actin differences between stages and cell types, paired within samples | atlas |

F5–F7 use all non-AML atlas cells. Standardized values are relative to each dataset's control PM cells.
Heatmap groups need ≥ 30 cells. F8 sample means need ≥ 20 cells per group.
Signed-rank tests need ≥ 6 samples. Each AML arm is one library of pooled replicates, so an across-replicate
test is not possible for these arms.

In [ ]:
import re

import anndata as ad
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns

from IPython.display import display
from matplotlib.lines import Line2D
from matplotlib.patches import Patch, PathPatch
from matplotlib.path import Path as MplPath
from scipy.stats import false_discovery_control, wilcoxon

from settings import (
    AML_CLUSTERS,
    AML_DATASET,
    ATLAS_FILE,
    INTEGRATION_FILE,
    LABEL_TABLE,
    RESULTS_DIR,
    SEED,
    UNASSIGNED,
)

## Settings

The same cluster order and colours are used throughout. These settings also fix which cell groups are shown
and how many cells or samples a group needs.

In [ ]:
OUT_DIR = RESULTS_DIR / "figures"
TABLE_DIR = OUT_DIR / "tables"
PROFILE_DIR = RESULTS_DIR / "cluster_profiles"
CONTRAST_DIR = RESULTS_DIR / "leukemia_vs_control"

COLOURS = {
    "PM1": "#7AD323",
    "PM2": "#F3A324",
    "PM3": "#407402",
    "PM4": "#EFDD0C",
    "PM5": "#4EE1C1",
    "PM6": "#494848",
    "PM7": "#4A8FE1",
    "PM8": "#999999",
    "PM9": "#000000",
    "CC1": "#F6C9CD",
    "CC2": "#CF001A",
    "LM1": "#B795FD",
    "LM2": "#BB10DF",
}

CELLTYPES = ["PM", "LM", "PH", "CC"]
CONDITIONS = ["control", "wasp", "injury", "selected_uninfected"]
CONDITION_NAMES = {
    "control": "control",
    "wasp": "wasp-infested",
    "injury": "injured",
    "selected_uninfected": "resistance-selected, uninfected",
}

CONDITION_COLOURS = {
    "control": "#999999",
    "wasp": "#e08a2c",
    "injury": "#7a4fa3",
    "selected_uninfected": "#3e9c6a",
}

DATASETS = ["10x_w1118", "Cattenoz", "Dropseq_OreR", "Fu", "Girard", "Leitao", "Tattikota"]
DATASET_NAMES = {"10x_w1118": "10x w1118", "Dropseq_OreR": "Drop-seq", "Leitao": "Leitão"}
DATASET_COLOURS = {
    "10x_w1118": "#1f77b4",
    "Cattenoz": "#ffbb78",
    "Dropseq_OreR": "#2ca02c",
    "Fu": "#8c564b",
    "Girard": "#f7b6d2",
    "Leitao": "#bcbd22",
    "Tattikota": "#17becf",
}

In [ ]:
# Cell groups and the minimum evidence needed to show or test them.
MIN_CELLS = 30                 # cells required for a heatmap group
MIN_PAIR_CELLS = 20            # cells required per side of a sample pair
MIN_TEST_SAMPLES = 6           # paired samples required for a signed-rank test
ACTINS = ["Act5C", "Act42A"]    # mean log1p CP10k of these genes defines the actin value
ALL_ACTINS = ["Act5C", "Act42A", "Act57B", "Act79B", "Act87E", "Act88F"]

# Published labels that take precedence over the broad Yoon cell types.
INTERMEDIATE = {"Tattikota": ["LM1"], "Dropseq_OreR": ["LM 1"], "Leitao": ["LAM1"]}
STAGES = ["PH", "PM", "intermediate", "NimC1+ PM", "LM"]
# stages compared with NimC1-negative PM, in panel order
STAGE_COMPARISONS = ["PH", "intermediate", "NimC1+ PM", "LM"]

DOT_GENES = {
    "plasmatocyte": ["Hml", "Pxn", "NimC1", "eater", "Ppn", "vkg", "Col4a1"],
    "lamellocyte": ["atilla", "PPO3", "mys"],
    "crystal cell": ["PPO1", "PPO2", "lz"],
}

Z_LABEL = "z vs the dataset's control PM"
DRAWING_STYLE = {
    "svg.fonttype": "none",
    "pdf.fonttype": 42,
    "font.size": 8,
    "axes.titlesize": 9,
    "font.family": "sans-serif",
    "font.sans-serif": ["Arial", "Liberation Sans", "DejaVu Sans"],
}

In [ ]:
# Choices that affect what the figures show
Z_SCORE_CAP = 8.0                       # colour scale of the z-score heatmaps stops here (numbers are not capped)
PROFILE_COLOUR_LIMIT = 0.6              # colour scale of the correlation heatmaps (F2) runs from -0.6 to 0.6
HIGHLIGHT_CLUSTER = "PM4"               # drawn last in F0, so its points stay visible
UMAP_ROTATION_DEGREES = 90              # F0 shows the layout turned clockwise by this angle
MIN_SOURCE_CELLS = 20                   # F1 shows an AML label only if this many non-AML cells carry it
SOURCE_LABEL_MIN_PERCENT = 6            # F1 names a segment only if it is at least this large

# Colours used in several figures
MISSING_GROUP_COLOUR = "#e6e6e6"        # groups with too few cells
DIVERGING_CMAP = "RdBu_r"               # values that can be positive or negative
EXPRESSION_CMAP = "viridis"             # values that start at zero
PAIRED_POSITIVE_COLOUR = "#c43c39"      # "higher in the first group" in F8
PAIRED_NEGATIVE_COLOUR = "#4a7fb5"      # "higher in the second group" in F8

In [ ]:
TABLE_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(context="paper", style="ticks", rc=DRAWING_STYLE)

## Save each figure in three formats
Re-running the notebook overwrites the named figures.
The helper closes each figure after saving so the full set can be drawn in one session.

In [ ]:
def save(figure, name):
    """Save SVG with live text, PDF and 300-dpi PNG."""

    for suffix in ["svg", "pdf", "png"]:
        options = {"dpi": 300} if suffix == "png" else {}
        figure.savefig(OUT_DIR / f"{name}.{suffix}", bbox_inches="tight", **options)

    plt.show()
    plt.close(figure)

## Draw a simple heatmap
The helper scales the panel to the number of rows and columns, with no cell borders.
Vertical lines separate datasets in the profile correlations.

In [ ]:
def heatmap(
    table,
    title,
    name,
    cmap=DIVERGING_CMAP,
    center=None,
    vmin=None,
    vmax=None,
    label="",
    width_per_column=0.42,
    height_per_row=0.28,
    separators=None,
):
    """Heatmap with a labelled colour scale and optional dataset separators."""
    data = table.astype(float)
    width = max(3.5, width_per_column * data.shape[1] + 2)
    height = max(2.5, height_per_row * data.shape[0] + 1)

    figure, axis = plt.subplots(figsize=(width, height))

    sns.heatmap(
        data,
        cmap=cmap,
        center=center,
        vmin=vmin,
        vmax=vmax,
        linewidths=0,
        cbar_kws={"label": label, "shrink": 0.6},
        ax=axis,
    )

    for position in separators or []:
        axis.axvline(position, color="black", lw=0.8)

    axis.set(title=title, xlabel="", ylabel="")
    axis.set_yticks(axis.get_yticks(), axis.get_yticklabels(), rotation=0)
    axis.tick_params(axis="both", length=0)
    save(figure, name)

## Draw dataset and condition heatmaps
F5–F7 show group means, with grey for groups below the cell threshold.
A coloured strip identifies the condition, and white gaps separate datasets.

In [ ]:
def group_heatmap(table, title, name, label, centered=True, cap=None, decimals=1, cmap=None):
    """Annotated group means with a condition strip and optional capped diverging scale."""
    rows = []

    for dataset in DATASETS:

        for condition in CONDITIONS:

            if (dataset, condition) in table.index:
                rows.append((dataset, condition))

    data = table.reindex(index=rows, columns=CELLTYPES).dropna(how="all").astype(float)
    peak = np.nanmax(np.abs(data.to_numpy()))

    if centered:
        limit = min(peak, cap) if cap else peak
        scale = dict(
            cmap=cmap or DIVERGING_CMAP,
            center=0,
            vmin=-limit,
            vmax=limit,
        )

    else:
        scale = dict(cmap=cmap or EXPRESSION_CMAP, vmin=0, vmax=peak)

    # Format rounded values; adding zero avoids displaying negative zero.
    shown = (data.round(decimals) + 0.0).map(
        lambda value: "" if np.isnan(value) else f"{value:.{decimals}f}",
    )

    figure, axes = plt.subplots(
        1,
        2,
        figsize=(5.2, 0.32 * len(data) + 1.2),
        gridspec_kw={"width_ratios": [0.06, 1], "wspace": 0.02},
    )

    strip = axes[0]
    axis = axes[1]
    conditions = [condition for dataset, condition in data.index]
    strip_colours = [[mcolors.to_rgb(CONDITION_COLOURS[condition])] for condition in conditions]

    strip.imshow(np.array(strip_colours), aspect="auto", extent=(0, 1, len(data), 0))

    row_labels = []

    for dataset, condition in data.index:
        row_labels.append(f"{DATASET_NAMES.get(dataset, dataset)} · {CONDITION_NAMES[condition]}")

    strip.set_xticks([])
    strip.set_yticks(np.arange(len(data)) + 0.5, row_labels)
    strip.set_ylim(len(data), 0)
    strip.tick_params(axis="y", length=0)

    # Grey background remains visible where a group has too few cells.
    axis.set_facecolor(MISSING_GROUP_COLOUR)
    extend = "both" if centered and cap and peak > cap else "neither"
    sns.heatmap(
        data,
        annot=shown,
        fmt="",
        annot_kws={"fontsize": 6.5},
        linewidths=0,
        yticklabels=False,
        ax=axis,
        cbar_kws={"label": label, "shrink": 0.7, "extend": extend},
        **scale,
    )

    datasets = [dataset for dataset, condition in data.index]

    for position in range(1, len(datasets)):

        if datasets[position] != datasets[position - 1]:

            for target in [strip, axis]:
                target.axhline(position, color="white", lw=2.5)

    axis.set(title=title, xlabel="cell type", ylabel="")
    axis.tick_params(axis="x", length=0)
    sns.despine(ax=strip, left=True, bottom=True)
    sns.despine(ax=axis, left=True, bottom=True)

    legend = []

    for condition in CONDITIONS:

        if condition in conditions:
            legend.append(Patch(color=CONDITION_COLOURS[condition], label=CONDITION_NAMES[condition]))

    legend.append(Patch(color=MISSING_GROUP_COLOUR, label=f"< {MIN_CELLS} cells"))
    axis.legend(
        handles=legend,
        loc="upper left",
        bbox_to_anchor=(1.35, 1.0),
        frameon=False,
        fontsize=7,
    )
    save(figure, name)

## Draw the sources of transferred AML labels
The bars stack dataset and published-label percentages for each AML label.
Large segments carry text; lighter shades distinguish labels from the same dataset.

In [ ]:
def source_bars(percent, title, name):
    """Horizontal source bars, with names on segments of at least 6%."""
    shade = {}

    for dataset in DATASETS:
        sources = [source for source in percent.columns if source.split(": ")[0] == dataset]
        sources = sorted(sources, key=lambda source: -percent[source].sum())
        rgb = np.array(mcolors.to_rgb(DATASET_COLOURS[dataset]))

        for index, source in enumerate(sources):
            shade[source] = tuple(
                rgb + (1 - rgb) * min(0.75, 0.12 * index),
            )

    def source_order(source):
        dataset = source.split(": ")[0]

        return DATASETS.index(dataset), -percent[source].sum()

    columns = sorted(percent.columns, key=source_order)
    figure, axis = plt.subplots(
        figsize=(8.5, 0.34 * len(percent) + 1.3),
    )
    left = np.zeros(len(percent))
    y = np.arange(len(percent))

    for source in columns:
        values = percent[source].fillna(0).to_numpy()

        axis.barh(y, values, left=left, color=shade[source], edgecolor="white", linewidth=0.3)

        dataset, label = source.split(": ", 1)

        for row, value in enumerate(values):

            if value >= SOURCE_LABEL_MIN_PERCENT:
                text = f"{DATASET_NAMES.get(dataset, dataset)}\n{label}"
                axis.text(
                    left[row] + value / 2,
                    row,
                    text,
                    ha="center",
                    va="center",
                    fontsize=5.5,
                    linespacing=0.95,
                )

        left += values

    axis.set_yticks(y, percent.index)
    axis.invert_yaxis()
    axis.set_xlim(0, 100)
    axis.set(xlabel="% of the label's non-AML cells", title=title)

    present = [dataset for dataset in DATASETS if dataset in {source.split(": ")[0] for source in columns}]
    legend = []

    for dataset in present:
        legend.append(Patch(color=DATASET_COLOURS[dataset], label=DATASET_NAMES.get(dataset, dataset)))

    axis.legend(
        handles=legend,
        fontsize=7,
        frameon=False,
        loc="upper left",
        bbox_to_anchor=(1.0, 1.0),
    )
    sns.despine(ax=axis)
    save(figure, name)

## Draw UMAP points as editable paths
The helper combines the small hexagons for each label into one vector object.
The grid and marker radius keep the appearance consistent with the panel's physical size.

In [ ]:
angles = np.linspace(0, 2 * np.pi, 7)
HEXAGON = np.c_[np.cos(angles), np.sin(angles)][:-1]


def dot_layer(axis, xy, colour, name, points=1.6):
    """Draw one label as a compound path of small hexagons."""
    axis.figure.canvas.draw()
    transformed = axis.transData.transform([(0, 0), (1, 1)])
    origin = transformed[0]
    unit = transformed[1]
    pixels = points / 72 * axis.figure.dpi / 2

    radius = np.array([pixels / abs(unit[0] - origin[0]), pixels / abs(unit[1] - origin[1])])
    grid = np.unique(np.round(xy / (radius / 2)), axis=0) * (radius / 2)

    vertices = grid[:, None, :] + radius * HEXAGON[None, :, :]
    vertices = np.concatenate([vertices, vertices[:, :1, :]], axis=1).reshape(-1, 2)

    codes = np.tile(
        [MplPath.MOVETO] + [MplPath.LINETO] * 5 + [MplPath.CLOSEPOLY],
        len(grid),
    )

    patch = PathPatch(MplPath(vertices, codes), facecolor=colour, edgecolor="none", linewidth=0)
    patch.set_gid(re.sub(r"[^A-Za-z0-9_+-]", "_", name))

    axis.add_patch(patch)

## Standardize values against each dataset's control PM cells
A z-score subtracts the mean of the control plasmatocytes (PM) and divides by their sample standard
deviation.
Each dataset uses its own control PM cells, so zero means its own control PM average and one is one standard
deviation above that average. This accounts for different expression baselines between studies.
Values are then summarized by dataset, condition and Yoon cell type; groups below 30 cells are hidden.

In [ ]:
def within_dataset_z(values, obs):
    """Z-score relative to control plasmatocytes in the same dataset."""
    values = pd.Series(values, index=obs.index, dtype=float)
    standardized = pd.Series(np.nan, index=obs.index)

    for dataset in obs.loc[obs["dataset"].ne(AML_DATASET), "dataset"].unique():
        in_dataset = obs["dataset"].eq(dataset).to_numpy()
        is_pm = obs["yoon_celltype"].eq("PM").to_numpy()
        is_control = obs["condition"].eq("control").to_numpy()

        reference = in_dataset & is_pm & is_control
        standardized[in_dataset] = (values[in_dataset] - values[reference].mean()) / values[reference].std()

    return standardized


def by_dataset_condition(values, obs):
    """Mean by dataset, condition and cell type; groups below 30 cells are missing."""
    frame = obs.assign(value=values)
    frame = frame[frame["dataset"].ne(AML_DATASET) & frame["yoon_celltype"].isin(CELLTYPES)]
    grouped = frame.groupby(["dataset", "condition", "yoon_celltype"], observed=True)["value"]

    return grouped.mean().where(grouped.size() >= MIN_CELLS).unstack("yoon_celltype")

## Label the paired-sample tests
Stars describe the unadjusted Wilcoxon p-value displayed above each dataset.
The saved tables also include Benjamini–Hochberg q-values across each figure's tests.

In [ ]:
def stars(pvalue):
    """Readable significance labels; missing tests have no stars."""

    if pd.isna(pvalue):
        return None

    if pvalue < 0.001:
        return "***"

    if pvalue < 0.01:
        return "**"

    if pvalue < 0.05:
        return "*"

    return "n.s."

## Draw paired-sample differences
Each dot is a difference between two groups in the same sample; a bar marks the dataset median.
Dataset order and seeded horizontal jitter stay consistent across the panels.

In [ ]:
def paired_panels(differences, tests, title, name, ylabel):
    """One panel per comparison, with sample dots, dataset medians and signed-rank p-values."""
    comparisons = list(differences)
    limit = 1.25 * max((block["difference"].abs().max() for block in differences.values()))

    rng = np.random.default_rng(SEED)

    figure, axes = plt.subplots(
        1,
        len(comparisons),
        figsize=(4.0 * len(comparisons), 4.4),
        sharey=True,
        squeeze=False,
    )

    for axis, comparison in zip(axes[0], comparisons):
        first, second = comparison.split(" vs ")
        block = differences[comparison]
        present = [dataset for dataset in DATASETS if dataset in set(block["dataset"])]

        # Red and blue halves show which group has the higher actin value.
        axis.axhspan(0, limit, color=PAIRED_POSITIVE_COLOUR, alpha=0.07, zorder=0)
        axis.axhspan(-limit, 0, color=PAIRED_NEGATIVE_COLOUR, alpha=0.07, zorder=0)
        axis.axhline(0, color="black", lw=1.2, zorder=1)

        for position, dataset in enumerate(present):
            values = block.loc[block["dataset"].eq(dataset), "difference"].to_numpy()
            x = position + rng.uniform(-0.15, 0.15, len(values))

            axis.scatter(
                x,
                values,
                s=34,
                color=DATASET_COLOURS[dataset],
                edgecolor="black",
                linewidth=0.5,
                zorder=3,
            )
            axis.hlines(
                np.median(values),
                position - 0.3,
                position + 0.3,
                color="black",
                lw=2,
                zorder=4,
            )

            pvalue = tests.get((comparison, dataset), np.nan)
            annotation = f"{stars(pvalue)}\n{pvalue:.1g}" if stars(pvalue) else f"n={len(values)}"
            axis.text(
                position,
                0.97 * limit,
                annotation,
                ha="center",
                va="top",
                fontsize=6.5,
            )

        axis.text(
            0.02,
            0.80,
            f"higher in {first}",
            transform=axis.transAxes,
            fontsize=7.5,
            color=PAIRED_POSITIVE_COLOUR,
        )
        axis.text(
            0.02,
            0.03,
            f"higher in {second}",
            transform=axis.transAxes,
            fontsize=7.5,
            color=PAIRED_NEGATIVE_COLOUR,
        )

        labels = [DATASET_NAMES.get(dataset, dataset) for dataset in present]
        axis.set_xticks(
            range(len(present)),
            labels,
            rotation=45,
            ha="right",
            rotation_mode="anchor",
        )
        axis.set(title=comparison, ylim=(-limit, limit))
        sns.despine(ax=axis)

    axes[0, 0].set_ylabel(ylabel)
    subtitle = (
        f"{title}; Wilcoxon signed-rank p over samples "
        f"(datasets with ≥ {MIN_TEST_SAMPLES} samples; n = samples otherwise)"
    )
    figure.suptitle(subtitle, fontsize=8, y=1.01)
    save(figure, name)

## F0: prepare the circulating-cell UMAP
The transferred labels are read and the fixed layout is turned 90° clockwise.
Unassigned cells are hidden, while the full layout's limits keep the picture at the same scale.

In [ ]:
side = ad.read_h5ad(INTEGRATION_FILE, backed="r")
label_table = pd.read_csv(LABEL_TABLE, sep="\t", index_col=0)
labels = label_table.loc[side.obs_names, "aml_label"].astype(str).to_numpy()

angle = np.deg2rad(UMAP_ROTATION_DEGREES)
turn = np.array([[np.cos(angle), -np.sin(angle)], [np.sin(angle), np.cos(angle)]])
xy = np.asarray(side.obsm["X_umap"]) @ turn

shown = labels != UNASSIGNED
span = np.ptp(xy, axis=0) + 1.0
figure, axis = plt.subplots(
    figsize=(4.4 * span[0] / span[1] + 1.0, 4.4),
)
axis.set_xlim(xy[:, 0].min() - 0.5, xy[:, 0].max() + 0.5)
axis.set_ylim(xy[:, 1].min() - 0.5, xy[:, 1].max() + 0.5)

axis.set_aspect("equal")
axis.set_xticks([])
axis.set_yticks([])
axis.set(xlabel="UMAP 1", ylabel="UMAP 2")
sns.despine(ax=axis)

## F0: draw the labels
Larger groups are drawn first and PM4 last so its points remain visible.
The legend follows the common AML cluster order.

In [ ]:
present = [cluster for cluster in AML_CLUSTERS if (labels == cluster).any()]
background = [cluster for cluster in present if cluster != HIGHLIGHT_CLUSTER]
draw_order = sorted(background, key=lambda cluster: -(labels == cluster).sum()) + [HIGHLIGHT_CLUSTER]

for cluster in draw_order:
    dot_layer(axis, xy[shown & (labels == cluster)], COLOURS[cluster], cluster, points=0.7)

legend = []

for cluster in present:
    legend.append(
        Line2D(
            [],
            [],
            marker="o",
            linestyle="",
            markersize=5,
            markerfacecolor=COLOURS[cluster],
            markeredgecolor="none",
            label=cluster,
        ),
    )

axis.legend(
    handles=legend,
    loc="upper left",
    bbox_to_anchor=(1.01, 1.0),
    frameon=False,
    fontsize=7,
)

unassigned_count = int((~shown).sum())
axis.set_title(
    f"AML cluster labels of the circulating cells; {unassigned_count:,} unassigned cells not shown",
)
save(figure, "F0_umap_aml_labels")

print(f"{unassigned_count:,} unassigned circulating cells are not shown in F0") 

display(figure)

## F1: show where the transferred labels come from
Non-AML circulating cells are counted by AML label and by dataset : published label.
Unassigned cells and AML labels represented by fewer than 20 non-AML cells are left out.

In [ ]:
atlas_metadata = ad.read_h5ad(ATLAS_FILE, backed="r")
obs_trial = atlas_metadata.obs.loc[side.obs_names, ["dataset", "published_label"]].astype("string")
atlas_metadata.file.close()
side.file.close()

frame = obs_trial.assign(label=labels)
frame = frame[frame["dataset"].ne(AML_DATASET) & frame["label"].ne(UNASSIGNED)]

published = frame["published_label"].fillna("no published label")
frame = frame.assign(source=frame["dataset"] + ": " + published)
rows = [cluster for cluster in AML_CLUSTERS if (frame["label"] == cluster).sum() >= MIN_SOURCE_CELLS]
percent = 100 * pd.crosstab(frame["label"], frame["source"], normalize="index").reindex(rows)

percent.round(2).to_csv(TABLE_DIR / "F1_source_composition_non_aml.tsv", sep="\t")

source_bars(
    percent,
    "Non-AML cells of each AML label by dataset : published label",
    "F1_source_composition",
)

## F2: compare relative expression profiles
A relative profile is a group's mean expression minus its own dataset's mean expression. The correlations
compare these patterns for the common Yoon labels and each study's published labels.
Dataset separators and 0.2 inches per column leave room to read the reference labels in both panels.

In [ ]:
profile_titles = {
    "yoon_labels": "Relative-profile Spearman correlation: AML clusters vs the other datasets' cell types",
    "published_labels": (
        "Relative-profile Spearman correlation: AML clusters vs the other datasets' published clusters"
    ),
}

for scheme, title in profile_titles.items():
    table = pd.read_csv(
        PROFILE_DIR / f"profile_correlation_{scheme}.tsv",
        sep="\t",
        header=[0, 1],
        index_col=0,
    )

    datasets = table.columns.get_level_values(0)
    separators = [i for i in range(1, len(datasets)) if datasets[i] != datasets[i - 1]]
    table.columns = [f"{DATASET_NAMES.get(dataset, dataset)}: {label}" for dataset, label in table.columns]
    table = table.reindex([cluster for cluster in AML_CLUSTERS if cluster in table.index])

    name = "F2_profile_correlation"

    if scheme == "published_labels":
        name = "F2_profile_correlation_published_labels"

    heatmap(
        table,
        title,
        name,
        center=0,
        vmin=-PROFILE_COLOUR_LIMIT,
        vmax=PROFILE_COLOUR_LIMIT,
        label="Spearman r",
        width_per_column=0.2,
        separators=separators,
    )

## F3: show the strongest shared leukemia changes
The top 30 shared genes retain their ranking and their whole-arm then within-cluster column order.
A symmetric colour scale covers all plotted log2 fold changes, so increases and decreases are comparable.

In [ ]:
table = pd.read_csv(CONTRAST_DIR / "top_shared_genes_log2fc.tsv", sep="\t", index_col=0)
limit = np.ceil(np.nanmax(np.abs(table.to_numpy())))
heatmap(
    table,
    "Top 30 shared leukemia genes: log2FC vs Empty_control, whole arm and within cluster",
    "F3_shared_leukemia_genes",
    center=0,
    vmin=-limit,
    vmax=limit,
    label="log2FC",
    width_per_column=0.32,
    height_per_row=0.24,
)

## F4: select programme genes in AML clusters
Marker genes are located by symbol in the log1p CP10k expression matrix.
The dot plot uses all AML clusters in the common order, including the small PM9 cluster.

In [ ]:
atlas = ad.read_h5ad(ATLAS_FILE)
del atlas.layers["counts"]

symbols = atlas.var["symbol"].astype("string")

is_aml = atlas.obs["dataset"].eq(AML_DATASET).to_numpy()
is_cluster = atlas.obs["published_label"].astype(str).isin(AML_CLUSTERS).to_numpy()
aml_mask = is_aml & is_cluster

columns = {}

for names in DOT_GENES.values():

    for gene in names:
        columns[gene] = int(np.flatnonzero(symbols.eq(gene).to_numpy())[0])

aml_labels = atlas.obs.loc[aml_mask, "published_label"].astype(str)
order = [cluster for cluster in AML_CLUSTERS if (aml_labels == cluster).any()]

dot_obs = pd.DataFrame(
    {"cluster": pd.Categorical(aml_labels, categories=order)},
    index=atlas.obs_names[aml_mask],
)

dot_data = ad.AnnData(
    X=atlas.X[aml_mask][:, list(columns.values())],
    var=pd.DataFrame(index=list(columns)),
    obs=dot_obs,
)

## F4: draw the programme dot plot
Dot colour is scaled separately for each gene, and dot size shows the detected fraction.
Gene labels are angled so they remain readable.

In [ ]:
dot = sc.pl.dotplot(
    dot_data,
    var_names=DOT_GENES,
    groupby="cluster",
    standard_scale="var",
    cmap=EXPRESSION_CMAP,
    return_fig=True,
)
dot.make_figure()

for axis in dot.fig.axes:

    for label in axis.get_xticklabels():
        label.set_rotation(45)
        label.set_ha("right")
        label.set_rotation_mode("anchor")

save(dot.fig, "F4_programme_genes_dotplot")

## Prepare metadata and single-gene expression
The condition column identifies the experimental groups; Yoon cell types define the non-AML summaries.
The expression helper reads a gene's log1p CP10k values by symbol.

In [ ]:
obs_columns = ["dataset", "sample_id", "yoon_celltype", "published_label", "condition"]
obs = atlas.obs[obs_columns].astype("string")
obs["yoon_celltype"] = obs["yoon_celltype"].fillna("none")


def expression(gene):
    """Log1p CP10k expression of a gene identified by its symbol."""
    column = int(np.flatnonzero(symbols.eq(gene).to_numpy())[0])

    return np.asarray(atlas.X[:, column].todense()).ravel()

## F5: score the PM4 signature
The score uses every signature gene in file order and expression-matched control genes chosen with a fixed
seed.
The panel shows both the score and its z-score relative to the same dataset's control PM cells.

In [ ]:
pm4_markers = pd.read_csv(PROFILE_DIR / "pm4_signature_genes.tsv", sep="\t", index_col=0)
sc.tl.score_genes(atlas, gene_list=list(pm4_markers.index), score_name="pm4_signature", random_state=SEED)

signature = atlas.obs["pm4_signature"].to_numpy()
signature_standardized = within_dataset_z(signature, obs).to_numpy()
signature_z = by_dataset_condition(signature_standardized, obs)
signature_score = by_dataset_condition(signature, obs)

signature_z.round(3).to_csv(TABLE_DIR / "F5a_pm4_signature_z.tsv", sep="\t")
signature_score.round(4).to_csv(TABLE_DIR / "F5b_pm4_signature_score.tsv", sep="\t")

group_heatmap(signature_z, "PM4 signature (z)", "F5a_pm4_signature_z", Z_LABEL, cap=Z_SCORE_CAP)
group_heatmap(
    signature_score,
    "PM4 signature score",
    "F5b_pm4_signature_score",
    "score (score_genes)",
    decimals=2,
)

## F6: summarize all six actin genes
Means and detection percentages show the actin expression pattern in each cell group.
Groups need at least 30 cells. Act5C and Act42A define the actin value shown in the figures.

In [ ]:
actin_expression = {}

for gene in ALL_ACTINS:

    if symbols.eq(gene).any():
        actin_expression[gene] = expression(gene)

actins = pd.DataFrame(actin_expression, index=obs.index)
groups = {}

for celltype in CELLTYPES:
    groups[f"non-AML {celltype}"] = (
        obs["dataset"].ne(AML_DATASET).to_numpy() & obs["yoon_celltype"].eq(celltype).to_numpy()
    )

for cluster in AML_CLUSTERS:
    groups[f"AML {cluster}"] = (
        obs["dataset"].eq(AML_DATASET).to_numpy() & obs["published_label"].eq(cluster).to_numpy()
    )

summaries = {}

for name, mask in groups.items():

    if mask.sum() >= MIN_CELLS:
        means = actins[mask].mean().add_suffix(" mean")
        detected = 100 * (actins[mask] > 0).mean().add_suffix(" detected %")
        summaries[name] = pd.concat([means, detected])

actin_table = pd.DataFrame(summaries).T
actin_table.round(3).to_csv(TABLE_DIR / "F6_actin_genes_by_group.tsv", sep="\t")
display(actin_table.round(2).head(8))

## F6: draw actin expression
Act5C and Act42A log1p CP10k are averaged per cell, then summarized by dataset and condition.
The z-score panel caps its colour scale at 8 while retaining the exact annotations.

In [ ]:
obs["actin"] = actins[ACTINS].mean(axis=1)
actin_values = obs["actin"].to_numpy()
actin_standardized = within_dataset_z(actin_values, obs).to_numpy()
actin_z = by_dataset_condition(actin_standardized, obs)
actin_mean = by_dataset_condition(actin_values, obs)

actin_z.round(3).to_csv(TABLE_DIR / "F6a_actin_z.tsv", sep="\t")
actin_mean.round(4).to_csv(TABLE_DIR / "F6b_actin_mean.tsv", sep="\t")

group_heatmap(actin_z, "Actin, Act5C + Act42A (z)", "F6a_actin_z", Z_LABEL, cap=Z_SCORE_CAP)
group_heatmap(
    actin_mean,
    "Actin, Act5C + Act42A",
    "F6b_actin_mean",
    "mean log1p CP10k",
    centered=False,
    decimals=2,
)

## F7: draw pyr expression
The panel shows the mean log1p CP10k and the z-score against control PM cells.
Detection percentages are saved too, because low control detection can produce large z-scores.

In [ ]:
pyr = expression("pyr")
pyr_standardized = within_dataset_z(pyr, obs).to_numpy()
pyr_z = by_dataset_condition(pyr_standardized, obs)
pyr_mean = by_dataset_condition(pyr, obs)
pyr_detected = by_dataset_condition(100 * (pyr > 0).astype(float), obs)

pyr_z.round(3).to_csv(TABLE_DIR / "F7a_pyr_z.tsv", sep="\t")
pyr_mean.round(4).to_csv(TABLE_DIR / "F7b_pyr_mean.tsv", sep="\t")
pyr_detected.round(2).to_csv(TABLE_DIR / "F7_pyr_detected_percent.tsv", sep="\t")

group_heatmap(pyr_z, "pyr expression (z)", "F7a_pyr_z", Z_LABEL, cap=Z_SCORE_CAP)
group_heatmap(
    pyr_mean,
    "pyr expression",
    "F7b_pyr_mean",
    "mean log1p CP10k",
    centered=False,
    decimals=2,
)

## F8: assign stages in the lamellocyte progression
Plasmatocytes are split by detected NimC1 expression, while Yoon labels identify PH and LM.
Published intermediate labels take precedence over these broad cell types.

In [ ]:
non_aml = obs["dataset"].ne(AML_DATASET).to_numpy()
nonaml = obs[non_aml].copy()
nimc1 = expression("NimC1")[non_aml] > 0
nonaml["nimc1"] = nimc1
celltype = nonaml["yoon_celltype"].to_numpy()

stage = np.full(len(nonaml), None, dtype=object)
stage[(celltype == "PM") & ~nimc1] = "PM"
stage[(celltype == "PM") & nimc1] = "NimC1+ PM"
stage[celltype == "PH"] = "PH"
stage[celltype == "LM"] = "LM"

for dataset, published in INTERMEDIATE.items():
    in_dataset = nonaml["dataset"].eq(dataset).to_numpy()
    is_intermediate = nonaml["published_label"].isin(published).to_numpy()
    stage[in_dataset & is_intermediate] = "intermediate"

nonaml["stage"] = stage

## F8a: calculate each sample's stage means
Actin is averaged within each dataset, sample and stage, requiring at least 20 cells.
Paired differences use the NimC1-negative PM group from the same sample as the reference. Comparing groups
within a sample reduces the influence of differences between libraries or experimental conditions; each
sample
contributes one difference rather than treating its cells as independent biological replicates.

In [ ]:
grouped = nonaml[nonaml["stage"].notna()].groupby(
    ["dataset", "sample_id", "stage"],
    observed=True,
)["actin"]
stage_means = grouped.mean()[grouped.size() >= MIN_PAIR_CELLS].unstack("stage").reindex(columns=STAGES)
stage_means.round(4).to_csv(TABLE_DIR / "F8a_actin_stage_means_per_sample.tsv", sep="\t")

## Test paired differences across samples
The Wilcoxon signed-rank test is run for datasets with at least six paired samples.
All available p-values are adjusted together within each figure using Benjamini–Hochberg.

In [ ]:
def tested(differences):
    """Signed-rank p per comparison and dataset, with BH adjustment across the figure."""
    rows = []

    for comparison, block in differences.items():

        for dataset, part in block.groupby("dataset"):
            pvalue = np.nan

            if len(part) >= MIN_TEST_SAMPLES:
                pvalue = wilcoxon(part["difference"]).pvalue

            samples_up = int((part["difference"] > 0).sum())
            rows.append(
                dict(
                    comparison=comparison,
                    dataset=dataset,
                    samples=len(part),
                    samples_up=f"{samples_up}/{len(part)}",
                    median_difference=part["difference"].median(),
                    wilcoxon_p=pvalue,
                ),
            )

    table = pd.DataFrame(rows).set_index(["comparison", "dataset"])
    table["bh_q"] = np.nan

    has_test = table["wilcoxon_p"].notna()
    table.loc[has_test, "bh_q"] = false_discovery_control(table["wilcoxon_p"].dropna().to_numpy())

    return table

## F8a: draw stage differences from PM
Only samples with both a stage mean and a NimC1-negative PM mean contribute a paired difference.
The panel shows sample differences, dataset medians and the signed-rank p-values.

In [ ]:
differences = {}

for stage in STAGE_COMPARISONS:
    difference = (stage_means[stage] - stage_means["PM"]).dropna()
    differences[f"{stage} vs PM"] = difference.rename("difference").reset_index()

tests = tested(differences)
tests.round(5).to_csv(TABLE_DIR / "F8a_actin_vs_pm_tests.tsv", sep="\t")
display(tests.round(4).head(8))

paired_panels(
    differences,
    tests["wilcoxon_p"],
    "Actin (Act5C + Act42A): stage minus NimC1− PM of the same sample",
    "F8a_actin_vs_pm",
    "actin: stage minus PM, per sample (bar = median)",
)

## F8b: pair broad cell types within each sample
The comparisons are LM with all PM, PH with all PM, and NimC1-positive with NimC1-negative PM.
Each side needs at least 20 cells; these comparisons use Yoon types without intermediate-stage overrides.

In [ ]:
def paired(first, second):
    """Per-sample actin differences between two cell masks, with at least 20 cells per side."""
    rows = []

    for (dataset, sample), block in nonaml.groupby(["dataset", "sample_id"], observed=True):
        first_cells = block[first.loc[block.index]]
        second_cells = block[second.loc[block.index]]

        if len(first_cells) >= MIN_PAIR_CELLS and len(second_cells) >= MIN_PAIR_CELLS:
            difference = first_cells["actin"].mean() - second_cells["actin"].mean()

            rows.append(dict(dataset=dataset, sample_id=sample, difference=difference))

    return pd.DataFrame(rows)

## F8b: draw the paired cell-type comparisons
The comparison order stays fixed so the same seeded jitter is used in each panel.
Both the raw p-values and the adjusted q-values are saved alongside the figure.

In [ ]:
is_pm = nonaml["yoon_celltype"].eq("PM")
is_lm = nonaml["yoon_celltype"].eq("LM")
is_ph = nonaml["yoon_celltype"].eq("PH")
is_nimc1 = nonaml["nimc1"]

differences = {
    "LM vs PM": paired(is_lm, is_pm),
    "PH vs PM": paired(is_ph, is_pm),
    "NimC1+ PM vs NimC1- PM": paired(is_pm & is_nimc1, is_pm & ~is_nimc1),
}

tests = tested(differences)
tests.round(5).to_csv(TABLE_DIR / "F8b_actin_paired_tests.tsv", sep="\t")
display(tests.round(4).head(8))

paired_panels(
    differences,
    tests["wilcoxon_p"],
    "Actin (Act5C + Act42A): paired cell types of the same sample",
    "F8b_actin_paired",
    "actin difference per sample (bar = median)",
)